In [ ]:
import pathlib
import os
from typing import List

from datasets import load_dataset
import evaluate
import torch
from transformers import (
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)
    return pretrained_model


# ---- Config ----
args = {
    'train_dataset': 'iggy12345/xnli-en-ipa',
    'eval_dataset': 'iggy12345/xnli-en-ipa',
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 32,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda'
}

models_and_tokenizers = [
    {"model_type": "ipa",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/english_spanish_ipa_12_5_medium_50k_3epoch/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-ipa-number-preservation-merges.txt")},
    {"model_type": "normal",
     "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/english_spanish_normal_12_5_medium_50k_3epoch/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-eng-spa-normal-number-preservation-merges.txt")},
]

main_output_dir = pathlib.Path("./training_outputs_es2enmedium")
os.makedirs(main_output_dir, exist_ok=True)

for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]

    print(f"\n🔤 Loading {model_type.upper()} model and tokenizer...")

    output_dir = main_output_dir / f"output_{model_type}"
    os.makedirs(output_dir, exist_ok=True)

    # ---- Load Tokenizer ----
    vocab_path, merges_path = tokenizer_paths
    tokenizer = load_tokenizer(vocab_path, merges_path)

    # ---- Load model ----
    base_model = load_pretrained_model(pathlib.Path(model_path), args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    # ---- Load datasets ----
    train_dataset = load_dataset(args['train_dataset'], split="train", cache_dir=str(args['hf_cache_dir']))
    eval_dataset = load_dataset(args['eval_dataset'], split="validation", cache_dir=str(args['hf_cache_dir']))

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    def preprocess_function(examples):
        if model_type == "ipa":
            feature = flatten_multi_features(examples, ['premise-phoneme', 'hypothesis-phoneme'])
        else:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    train_encoded = train_dataset.map(preprocess_function, batched=True)
    eval_encoded = eval_dataset.map(preprocess_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    metric = evaluate.load("xnli", "en")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions, average="macro")
        hf_metrics["recall"] = recall_score(labels, predictions, average="macro")
        return hf_metrics

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        metric_for_best_model="precision",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_encoded,
        eval_dataset=eval_encoded,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"\n🚀 Training {model_type.upper()} model on ES → Evaluating on EN")
    trainer.train(resume_from_checkpoint=False)

    results = trainer.evaluate()
    print(f"\n✅ Evaluation results (ES→EN) for model type: {model_type}:\n{results}")


In [ ]:
import os
import pathlib
from typing import List
from datasets import load_dataset, concatenate_datasets
import evaluate
import torch
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Config ---
train_lang = 'es'
eval_lang = 'es'

BASE_DIR = pathlib.Path("/fs/scratch/PAS2836/ipa_gpt")
TOKENIZER_DIR = pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers")

CHECKPOINTS = {
    "ipa": BASE_DIR / "checkpoints/english_spanish_ipa_12_5_medium_50k_3epoch/ckpt.pt",
    "normal": BASE_DIR / "checkpoints/english_spanish_normal_12_5_medium_50k_3epoch/ckpt.pt",
}

TOKENIZERS = {
    "ipa": (
        TOKENIZER_DIR / "bpe-eng-spa-ipa-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-eng-spa-ipa-number-preservation-merges.txt",
    ),
    "normal": (
        TOKENIZER_DIR / "bpe-eng-spa-normal-number-preservation-vocab.json",
        TOKENIZER_DIR / "bpe-eng-spa-normal-number-preservation-merges.txt",
    ),
}

LANG_TO_DATASET = {
    "en": "iggy12345/xnli-en-ipa",
    "es": "iggy12345/xnli-es-ipa"
}

args = {
    'epochs': 3,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 16,
    'hf_cache_dir': pathlib.Path('cache'),
    'device': 'cuda',
}

def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    checkpoint = torch.load(path, map_location=device)
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    filtered = {k: v for k, v in state_dict.items()
                if k in model.state_dict() and v.shape == model.state_dict()[k].shape}
    model.load_state_dict({**model.state_dict(), **filtered})
    return model.to(device)

def flatten_multi_features(examples, features: List[str]) -> List[str]:
    if len(features) == 1:
        return examples[features[0]]
    sep = f'\n\n{eod_token}\n\n'
    return [sep.join([x or '' for x in items]) for items in zip(*[examples[f] for f in features])]

def get_fields(example, model_type):
    if model_type == "ipa" and "premise-phoneme" in example:
        return ["premise-phoneme", "hypothesis-phoneme"]
    return ["premise", "hypothesis"]

def load_and_preprocess(dataset_name, split, tokenizer, model_type):
    ds = load_dataset(dataset_name, split=split, cache_dir=str(args['hf_cache_dir']))

    def preprocess(examples):
        fields = get_fields(examples, model_type)
        features = flatten_multi_features(examples, fields)
        return tokenizer(features, truncation=True, max_length=args['context_size'])

    return ds.map(preprocess, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.from_numpy(logits).argmax(dim=-1)
    labels = torch.from_numpy(labels)

    correct = (preds == labels).sum().item()
    total = len(labels)
    accuracy = correct / total

    return {
        "accuracy": accuracy,
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0)
    }

# === Run both IPA and NORMAL models ===
for model_type in ['ipa', 'normal']:
    print(f"\n🔧 Running setup for {model_type.upper()} model")
    vocab_path, merges_path = TOKENIZERS[model_type]
    tokenizer = load_tokenizer(vocab_path, merges_path)
    base_model = load_pretrained_model(CHECKPOINTS[model_type], args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(args['device'])

    if train_lang == 'both':
        train_en = load_and_preprocess(LANG_TO_DATASET['en'], 'train', tokenizer, model_type)
        train_es = load_and_preprocess(LANG_TO_DATASET['es'], 'train', tokenizer, model_type)
        train_dataset = concatenate_datasets([train_en, train_es]).shuffle()
    else:
        train_dataset = load_and_preprocess(LANG_TO_DATASET[train_lang], 'train', tokenizer, model_type)

    if eval_lang == 'both':
        eval_en = load_and_preprocess(LANG_TO_DATASET['en'], 'validation', tokenizer, model_type)
        eval_es = load_and_preprocess(LANG_TO_DATASET['es'], 'validation', tokenizer, model_type)
        eval_dataset = concatenate_datasets([eval_en, eval_es])
    elif eval_lang == 'en':
        eval_dataset = load_and_preprocess(LANG_TO_DATASET['en'], 'validation', tokenizer, model_type)
    else:
        eval_dataset = load_and_preprocess(LANG_TO_DATASET['es'], 'validation', tokenizer, model_type)

    output_dir = pathlib.Path(f"./training_outputs_engspa/{train_lang}2{eval_lang}_{model_type}")
    output_dir.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        metric_for_best_model="precision",
        load_best_model_at_end=True,
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=500,
        logging_dir='./logs',
        fp16=True,
        disable_tqdm=False,
        warmup_ratio=0.3,
        save_safetensors=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    print(f"Training {model_type.upper()} model on {train_lang.upper()} → Evaluating on {eval_lang.upper()}")
    trainer.train()

    print(f"Final evaluation on {eval_lang.upper()} for model {model_type.upper()}")
    results = trainer.evaluate()
    print(results)



🔧 Running setup for IPA model
number of parameters: 353.24M


Map: 100%|██████████| 2490/2490 [00:00<00:00, 8920.07 examples/s]
/tmp/slurmtmp.1674951/ipykernel_269512/643655616.py:152: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Training IPA model on ES → Evaluating on ES


wandb: Currently logged in as: orugantikoundinya7 (orugantikoundinya7-ohio-state-buckeyes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
500,1.084400,1.054468,0.436948,0.482776,0.436948,0.424348
1000,0.979200,0.986532,0.516867,0.541948,0.516867,0.512227
1500,0.950000,1.012544,0.524096,0.562437,0.524096,0.511601
2000,0.935800,0.938981,0.553012,0.601683,0.553012,0.549148
2500,0.920500,0.915281,0.581124,0.593243,0.581124,0.581683
3000,0.903800,0.924122,0.563454,0.613274,0.563454,0.554798
3500,0.893900,0.898056,0.589157,0.592850,0.589157,0.584783
4000,0.895000,0.928562,0.561044,0.603008,0.561044,0.550549
4500,0.886800,0.947902,0.558635,0.615088,0.558635,0.545367
5000,0.878400,0.970290,0.556225,0.614684,0.556225,0.544304
